In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import VarianceThreshold
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# -----------------------------
# Experiment metadata
# -----------------------------
experiment_name = "exp05_variance_filtered_elasticnet_20260323"

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv(
    "/Users/maggie/Documents/GitHub/NIR-Spectroscopy-Moisture-Prediction-Using-Machine-Learning/data/train.csv",
    encoding="cp932"
)

test = pd.read_csv(
    "/Users/maggie/Documents/GitHub/NIR-Spectroscopy-Moisture-Prediction-Using-Machine-Learning/data/test.csv",
    encoding="cp932"
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# -----------------------------
# Define spectral columns
# -----------------------------
spectral_cols = [c for c in train.columns if c not in ['sample number','species number','樹種','含水率']]

X = train[spectral_cols].values
y = train['含水率'].values
X_test = test[spectral_cols].values

# -----------------------------
# Variance filtering
# -----------------------------
selector = VarianceThreshold(threshold=0.0001)

X = selector.fit_transform(X)
X_test = selector.transform(X_test)

print("Features after variance filtering:", X.shape[1])

# -----------------------------
# Scaling (needed for ElasticNet)
# -----------------------------
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# -----------------------------
# Cross-validation
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []

for train_idx, val_idx in kf.split(X_scaled):

    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = ElasticNet(
        alpha=1.0,
        l1_ratio=0.7,
        max_iter=100000,
        tol=1e-3,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_scores.append(rmse)

print("CV RMSE:", np.mean(rmse_scores))

# -----------------------------
# Train final model
# -----------------------------
model = ElasticNet(
    alpha=1.0,
    l1_ratio=0.7,
    max_iter=100000,
    tol=1e-3,
    random_state=42
)

model.fit(X_scaled, y)

# -----------------------------
# Predict test set
# -----------------------------
test_preds = model.predict(X_test_scaled)

print("Sample predictions:", test_preds[:10])

# -----------------------------
# Create submission
# -----------------------------
submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

print(submission.head())

# -----------------------------
# Save submission
# -----------------------------
os.makedirs("../submissions", exist_ok=True)

output_file = f"../submissions/{experiment_name}.csv"

submission.to_csv(output_file, index=False, header=False)

print("Submission saved to:", output_file)

# -----------------------------
# Verify saved file
# -----------------------------
check = pd.read_csv(output_file, header=None)
print(check.head())

Train shape: (1322, 1559)
Test shape: (550, 1558)
Features after variance filtering: 1555
CV RMSE: 24.104188464573703
Sample predictions: [189.19253879 178.27372659 170.08663421 163.7306607  154.76597593
 144.40209002 135.02569038 127.61741902 121.24574927 115.97675995]
   sample number         含水率
0             95  189.192539
1             96  178.273727
2             97  170.086634
3             98  163.730661
4             99  154.765976
Submission saved to: ../submissions/exp05_variance_filtered_elasticnet_20260323.csv
    0           1
0  95  189.192539
1  96  178.273727
2  97  170.086634
3  98  163.730661
4  99  154.765976
